## 1D Profile Analysis

This notebook provides a detailed analysis of the 1D azimuthal integration profiles generated by the model. It includes visualizations of the profiles and statistical metrics to compare peak displacement between the predicted and true profiles, as well as the Pearson cross-correlation coefficient to assess the similarity between the two profiles.

In [ ]:
import sys
import yaml
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from pyFAI.integrator.azimuthal import AzimuthalIntegrator
from pyFAI.detectors import detector_factory

from tqdm import tqdm

project_root = r""
sys.path.append(project_root)

In [ ]:
from src.utils import create_model
from src.data_pipeline import  DiffractionDataset
sns.set_theme(style="whitegrid")

In [ ]:
CONFIG_PATH = r"" 
MODEL_PATH = r""
HDF5_PATH = r""

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

PARAM_NAMES = ["dist", "poni1", "poni2", "rot1", "rot2", "rot3"]
PARAM_UNITS = ["m", "m", "m", "rad", "rad", "rad"]

In [ ]:
# Load model in inference mode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = create_model(config)

print(f"Loading model weights from {MODEL_PATH}")
state_dict = torch.load(MODEL_PATH, map_location=device)

if any(k.startswith('_orig_mod.') for k in state_dict.keys()):
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k[10:] if k.startswith('_orig_mod.') else k
        new_state_dict[name] = v
    state_dict = new_state_dict

model.load_state_dict(state_dict)
model.to(device)
model.eval();

print("Model loaded for inference.")

In [ ]:
# Run the inference
image_size = config['model'].get('image_size', 224)
test_dataset = DiffractionDataset(HDF5_PATH, 'test', image_size=image_size)
test_loader = DataLoader(test_dataset, batch_size=config['training']['batch_size'], shuffle=False)

all_preds = []
all_labels = []

print("Running inference on the test set...")
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        predictions = model(images)
        
        all_preds.append(predictions.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

preds_np = np.concatenate(all_preds)
labels_np = np.concatenate(all_labels)

print("Inference complete.")

In [ ]:
# Calculate azimuthal integration profiles based on the predicted and true parameters
import h5py
detector = detector_factory("Eiger2Cdte_1M")
wavelength = 0.5121261413149675e-10

pred_profiles = []
true_profiles = []
pred_qs = []
true_qs = []

print("Calculating azimuthal integration profiles...")
with h5py.File(HDF5_PATH, "r") as f:
    images = f["test"]["images"]

    for i in tqdm(range(len(labels_np))):
        image = np.array(images[i], dtype=np.float32)

        pred_ai = AzimuthalIntegrator(
            dist=preds_np[i, 0],
            poni1=preds_np[i, 1],
            poni2=preds_np[i, 2],
            rot1=preds_np[i, 3],
            rot2=preds_np[i, 4],
            rot3=preds_np[i, 5],
            detector=detector,
            wavelength=wavelength
        )
        true_ai = AzimuthalIntegrator(
            dist=labels_np[i, 0],
            poni1=labels_np[i, 1],
            poni2=labels_np[i, 2],
            rot1=labels_np[i, 3],
            rot2=labels_np[i, 4],
            rot3=labels_np[i, 5],
            detector=detector,
            wavelength=wavelength
        )

        pred_q, pred_profile = pred_ai.integrate1d(image, npt=1000)
        true_q, true_profile = true_ai.integrate1d(image, npt=1000)

        pred_qs.append(pred_q)
        true_qs.append(true_q)
        pred_profiles.append(pred_profile)
        true_profiles.append(true_profile)

pred_profiles = np.array(pred_profiles)
true_profiles = np.array(true_profiles)

print("Profile calculation complete.")

In [ ]:
# Plot the predicted vs true profiles for a sample
sample_idx = 0
plt.figure(figsize=(8, 5))
plt.plot(true_qs[sample_idx], true_profiles[sample_idx], label="True Profile")
plt.plot(pred_qs[sample_idx], pred_profiles[sample_idx], label="Predicted Profile")
plt.xlabel("q (1/Å)")
plt.ylabel("Intensity (a.u.)")
plt.title("Azimuthal Integration Profile Comparison")
plt.legend()
plt.show()


In [ ]:
import numpy as np

wavelength_A = 0.5121261413149675  # Angstroms
two_theta_deg = 18.0
theta_rad = np.deg2rad(two_theta_deg / 2.0)
q_center = (4.0 * np.pi / wavelength_A) * np.sin(theta_rad)

q_min = q_center - 0.1  # adjust width as needed
q_max = q_center + 0.1

print("q_center:", q_center, "q_min:", q_min, "q_max:", q_max)
print("pred_q range:", pred_qs[0].min(), pred_qs[0].max())

In [ ]:
# Calculate and compare average absolute peak displacement between predicted and true profiles for peak at 2theta ~25 degrees 
peak_displacements = []

for i in range(len(labels_np)):
    pred_q = pred_qs[i]
    true_q = true_qs[i]

    pred_mask = (pred_q >= q_min) & (pred_q <= q_max)
    true_mask = (true_q >= q_min) & (true_q <= q_max)

    pred_peak_pos = pred_q[pred_mask][np.argmax(pred_profiles[i][pred_mask])]
    true_peak_pos = true_q[true_mask][np.argmax(true_profiles[i][true_mask])]

    peak_displacements.append(np.abs(pred_peak_pos - true_peak_pos))

avg_peak_displacement = np.mean(peak_displacements)
print(f"Average absolute peak displacement: {avg_peak_displacement:.4f}")

In [ ]:
# Calculate peak displacement and Pearson cross-correlation coefficient for each profile
from scipy.stats import pearsonr
from scipy.interpolate import interp1d

peak_displacements = []
correlation_coeffs = []

for i in range(len(labels_np)):
    pred_q = pred_qs[i]
    true_q = true_qs[i]

    pred_mask = (pred_q >= q_min) & (pred_q <= q_max)
    true_mask = (true_q >= q_min) & (true_q <= q_max)

    if not np.any(pred_mask) or not np.any(true_mask):
        continue

    pred_peak_pos = pred_q[pred_mask][np.argmax(pred_profiles[i][pred_mask])]
    true_peak_pos = true_q[true_mask][np.argmax(true_profiles[i][true_mask])]
    peak_displacements.append(np.abs(pred_peak_pos - true_peak_pos))

    interp_true = interp1d(true_q, true_profiles[i], kind="linear", bounds_error=False, fill_value="extrapolate")
    true_on_pred_q = interp_true(pred_q)

    corr_coeff, _ = pearsonr(pred_profiles[i], true_on_pred_q)
    correlation_coeffs.append(corr_coeff)

avg_pearson_corr = np.mean(correlation_coeffs)
print(f"Average Pearson correlation coefficient: {avg_pearson_corr:.4f}")

In [ ]:
# Visualize the distribution of peak displacements and correlation coefficients
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(peak_displacements, bins=100, kde=True)
plt.title("Distribution of Peak Displacements")
plt.subplot(1, 2, 2)
sns.histplot(correlation_coeffs, bins=100, kde=True)
plt.title("Distribution of Pearson Correlation Coefficients")
plt.tight_layout()
plt.show()